<a href="https://colab.research.google.com/github/layal-abd/SNN-Spike-Injection/blob/main/snn_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SNN spike-injection experiments (COMP 500)

**How to use this notebook**
1. Runtime > Change runtime type > **T4 GPU**.
2. First time: **Runtime > Run all**
3. After a Colab restart: **Run all** again. Training cells skip models that are already finished (they leave a `.done` marker on Google Drive), so only the attack cells take time.
4. Everything is saved in **Google Drive > MyDrive > snn_project** (data files, models, attack results).



## 1. Setup: libraries, Google Drive, dataset, yes/no feature files

In [ ]:
!pip install -q snntorch
import os, shutil, torch, torchaudio
from google.colab import drive

drive.mount('/content/drive')
P = '/content/drive/MyDrive/snn_project'
os.makedirs(P, exist_ok=True)
os.makedirs('./data', exist_ok=True)

# to bring back any saved .pt files from Drive
for f in os.listdir(P):
    if f.endswith('.pt'):
        shutil.copy(os.path.join(P, f), f)

# to download the dataset
for s in ['training', 'validation', 'testing']:
    torchaudio.datasets.SPEECHCOMMANDS('./data', download=True, subset=s)

# to build the yes/no feature files
mel_tf = torchaudio.transforms.MelSpectrogram(sample_rate=16000, n_mels=40)

def build(subset):
    ds = torchaudio.datasets.SPEECHCOMMANDS('./data', download=False, subset=subset)
    X, y = [], []
    for i, p in enumerate(ds._walker):
        word = os.path.basename(os.path.dirname(p))
        if word not in ('yes', 'no'):
            continue
        wav, sr, label, *_ = ds[i]
        if wav.shape[1] < 16000:
            wav = torch.nn.functional.pad(wav, (0, 16000 - wav.shape[1]))
        X.append((mel_tf(wav) + 1e-6).log().squeeze(0))
        y.append(1 if label == 'yes' else 0)
    return torch.stack(X), torch.tensor(y)

for s in ['training', 'validation', 'testing']:
    fn = f'yesno_{s}.pt'
    if os.path.exists(fn):
        print('found', fn)
    else:
        X, y = build(s)
        torch.save((X, y), fn)
        shutil.copy(fn, P)
        print('built', fn, tuple(X.shape), 'yes:', int(y.sum()), 'no:', int((y == 0).sum()))

print('GPU available:', torch.cuda.is_available())
print('saved on Drive:', sorted(os.listdir(P)))

## 2. Model code (network, guard penalty, training function)
The training and validation data.

In [ ]:
import os, shutil, torch, torch.nn as nn, snntorch as snn
from snntorch import surrogate
from snntorch import functional as SF
from torch.utils.data import TensorDataset, DataLoader

P = '/content/drive/MyDrive/snn_project'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
GAMMA, THR = 0.7, 1.0

Xtr, ytr = torch.load('yesno_training.pt')
Xva, yva = torch.load('yesno_validation.pt')
mean = Xtr.mean(dim=(0, 2), keepdim=True)
std = Xtr.std(dim=(0, 2), keepdim=True)
norm = lambda X: ((X - mean) / std).clamp(-5, 5)
Xtr, Xva = norm(Xtr), norm(Xva)

class SNN(nn.Module):
    def __init__(self, n_in=40, n_hid=256, n_out=2, beta=0.9, thr=1.0):
        super().__init__()
        grad = surrogate.fast_sigmoid()
        self.fc1 = nn.Linear(n_in, n_hid)
        self.lif1 = snn.Leaky(beta=beta, threshold=thr, spike_grad=grad)
        self.fc2 = nn.Linear(n_hid, n_out)
        self.lif2 = snn.Leaky(beta=beta, threshold=thr, spike_grad=grad)
        self.n_hid, self.n_out = n_hid, n_out

    def run(self, x, mode='none'):
        mem1 = self.lif1.init_leaky(); mem2 = self.lif2.init_leaky()
        out, pen = [], 0.0
        for t in range(x.shape[2]):
            spk1, mem1 = self.lif1(self.fc1(x[:, :, t]), mem1)
            spk2, mem2 = self.lif2(self.fc2(spk1), mem2)
            out.append(spk2)
            if mode in ('hidden', 'all'):
                pen = pen + (torch.relu(mem1 - GAMMA * THR) ** 2).sum(1).mean()
            if mode == 'all':
                pen = pen + (torch.relu(mem2 - GAMMA * THR) ** 2).sum(1).mean()
        n = self.n_hid + (self.n_out if mode == 'all' else 0)
        return torch.stack(out), pen / (n * x.shape[2])

    def forward(self, x):
        return self.run(x)[0]

loss_fn = SF.ce_rate_loss()

def evaluate(m, X, y):
    m.eval(); correct = 0
    with torch.no_grad():
        for i in range(0, len(X), 256):
            xb, yb = X[i:i+256].to(device), y[i:i+256].to(device)
            correct += (m(xb).sum(0).argmax(1) == yb).sum().item()
    return correct / len(X)

def train_model(name, lam=0.0, mode='none', seed=0, epochs=30):
    torch.manual_seed(seed)
    m = SNN().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    ldr = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
    best = 0.0
    for ep in range(epochs):
        m.train()
        for xb, yb in ldr:
            xb, yb = xb.to(device), yb.to(device)
            spk, pen = m.run(xb, mode)
            loss = loss_fn(spk, yb) + lam * pen
            opt.zero_grad(); loss.backward(); opt.step()
        va = evaluate(m, Xva, yva)
        if va > best:
            best = va
            torch.save(m.state_dict(), name)
            shutil.copy(name, P)
        if (ep + 1) % 5 == 0:
            print(f'{name} | epoch {ep+1} | validation accuracy {va:.3f} | best so far {best:.3f}')
    return best

print('model code ready | using', device)

## 3. Train the baseline

In [ ]:
marker = os.path.join(P, 'baseline_seed0.pt.done')
if os.path.exists(marker):
    print('skip (already done): baseline_seed0.pt | best validation accuracy', open(marker).read())
else:
    best = train_model('baseline_seed0.pt', lam=0.0, mode='none', seed=0)
    open(marker, 'w').write(str(best))
    print('baseline best validation accuracy:', round(best, 3))

## 4. Train the guarded models
Hidden-only guard (lambda 0.1, 1, 10) and all-neuron guard (lambda 1, 10).

In [ ]:
runs = [
    ('guardedH_lam0.1.pt', 0.1, 'hidden'),
    ('guardedH_lam1.pt',   1.0, 'hidden'),
    ('guardedH_lam10.pt', 10.0, 'hidden'),
    ('guardedA_lam1.pt',   1.0, 'all'),
    ('guardedA_lam10.pt', 10.0, 'all'),
]
for name, lam, mode in runs:
    marker = os.path.join(P, name + '.done')
    if os.path.exists(marker):
        print('skip (already done):', name)
        continue
    best = train_model(name, lam=lam, mode=mode, seed=0)
    open(marker, 'w').write(str(best))
    print(f'DONE {name} | guard: {mode} | lambda = {lam} | best validation accuracy {best:.3f}')

## 5. Attack tools and loading the six models
No attack runs here.

In [ ]:
T, H = Xva.shape[2], 256

def run_cur(m, x, inj, pulse, record=False):
    mem1 = m.lif1.init_leaky(); mem2 = m.lif2.init_leaky()
    counts = torch.zeros(x.shape[0], 2, device=x.device)
    mems, spks = [], []
    for t in range(x.shape[2]):
        cur1 = m.fc1(x[:, :, t]) + inj[:, t, :] * pulse
        spk1, mem1 = m.lif1(cur1, mem1)
        spk2, mem2 = m.lif2(m.fc2(spk1), mem2)
        counts += spk2
        if record:
            mems.append(mem1); spks.append(spk1)
    if record:
        return counts, torch.stack(mems, 1), torch.stack(spks, 1)
    return counts, None, None

def attack(m, idx, mode, budget=200, pulse=1.0, seed=0):
    x = Xva[idx].to(device); B = len(idx)
    d = (m.fc2.weight[1] - m.fc2.weight[0]).detach()
    with torch.no_grad():
        zero = torch.zeros(B, T, H, device=device)
        _, mems, spks = run_cur(m, x, zero, 0.0, record=True)
        if mode == 'random':
            g = torch.Generator(device=device).manual_seed(seed)
            score = torch.rand(B, T, H, generator=g, device=device)
        else:
            if mode == 'random_yes':
                g = torch.Generator(device=device).manual_seed(seed)
                score = torch.rand(B, T, H, generator=g, device=device)
            else:
                score = mems.clone()
            score[spks > 0] = -1e9
            score[:, :, d <= 0] = -1e9
        order = score.reshape(B, -1).topk(budget, dim=1).indices
        inj = torch.zeros(B, T * H, device=device)
        crit = torch.full((B,), float('inf'))
        for k in range(budget + 1):
            if k > 0:
                inj.scatter_(1, order[:, k-1:k], 1.0)
            counts, _, _ = run_cur(m, x, inj.view(B, T, H), pulse)
            flipped = (counts[:, 1] > counts[:, 0]).cpu()
            crit[flipped & torch.isinf(crit)] = k
            if not torch.isinf(crit).any():
                break
    return crit

def predict(m):
    m.eval()
    with torch.no_grad():
        return torch.cat([m(Xva[i:i+256].to(device)).sum(0).argmax(1).cpu() for i in range(0, len(Xva), 256)])

def hidden_activity(m, idx):
    with torch.no_grad():
        _, _, spks = run_cur(m, Xva[idx].to(device), torch.zeros(len(idx), T, H, device=device), 0.0, record=True)
    return spks.sum((1, 2)).mean().item()

files = {'baseline': 'baseline_seed0.pt',
         'hidden-only lam0.1': 'guardedH_lam0.1.pt', 'hidden-only lam1': 'guardedH_lam1.pt',
         'hidden-only lam10': 'guardedH_lam10.pt',
         'all-neuron lam1': 'guardedA_lam1.pt', 'all-neuron lam10 (failed run)': 'guardedA_lam10.pt'}
nets = {}
for k, f in files.items():
    if not os.path.exists(f):
        shutil.copy(os.path.join(P, f), f)
    mm = SNN().to(device); mm.load_state_dict(torch.load(f)); mm.eval(); nets[k] = mm

ok = None
for k, mm in nets.items():
    p = predict(mm)
    print(f'{k:30s} validation accuracy: {(p == yva).float().mean().item():.3f}')
    c = (yva == 0) & (p == 0)
    ok = c if ok is None else (ok & c)
idx = ok.nonzero().squeeze(1)
print('shared correctly rejected "no" clips:', len(idx))

## 6. Run the attack (validation set)
Modes: `random` (any neuron and time step, E1), `random_yes` (random among yes-voting neurons that are not firing), `near` (the same pool ranked by membrane potential, E2). Pulse sizes: 1.0 x threshold (the report's definition) and 0.5 x threshold. Budget: up to 200 spikes.

In [ ]:
budgets = (0, 5, 10, 20, 50, 100, 200)
results = {}
for pulse in (1.0, 0.5):
    print(f'\n===== pulse = {pulse} x threshold =====')
    for k, mm in nets.items():
        print(f'--- {k} | hidden spikes per clip: {hidden_activity(mm, idx):.0f}')
        for mode in ('random', 'random_yes', 'near'):
            crit = attack(mm, idx, mode, budget=200, pulse=pulse)
            results[(pulse, k, mode)] = crit
            sr = {b: round((crit <= b).float().mean().item(), 3) for b in budgets}
            reach = [b for b in range(0, 201) if (crit <= b).float().mean().item() >= 0.25]
            print(f'   {mode:10s}', sr, '| budget for 25%:', reach[0] if reach else 'not reached by 200')
torch.save(results, os.path.join(P, 'attack_val_results.pt'))
print('saved attack_val_results.pt to Drive')

## 7. Diagnostic: where do the membrane potentials sit?
Measures only ( No tuning ). Comparing the guarded models with the baseline.

In [ ]:
print('model | median top potential (x theta) | share of yes-voting silent slots above 0.5 / 0.7 / 0.9 theta | hidden spikes per clip')
for k, mm in nets.items():
    d = (mm.fc2.weight[1] - mm.fc2.weight[0]).detach()
    with torch.no_grad():
        _, mems, spks = run_cur(mm, Xva[idx].to(device), torch.zeros(len(idx), T, H, device=device), 0.0, record=True)
    pool = (spks == 0) & (d > 0).view(1, 1, H)
    top = mems.masked_fill(~pool, float('-inf')).reshape(len(idx), -1).max(1).values
    vals = mems[pool]
    print(f'{k:30s} | {top.median().item():5.2f} | {(vals > 0.5).float().mean().item():.3f} / {(vals > 0.7).float().mean().item():.3f} / {(vals > 0.9).float().mean().item():.3f} | {spks.sum((1, 2)).mean().item():.0f}')

## 8. FINAL RUNS, part 1: train 5 models x 5 seeds
Protocol fixed before looking at any final result: models = baseline, hidden-only guard (lambda 0.1, 1, 10), all-neuron guard (lambda 1); seeds 0 to 4; checkpoint = best validation accuracy.

In [ ]:
# train 5 models x 5 seeds
final_models = [
    ('baseline',      0.0, 'none'),
    ('hidden_lam0.1', 0.1, 'hidden'),
    ('hidden_lam1',   1.0, 'hidden'),
    ('hidden_lam10', 10.0, 'hidden'),
    ('all_lam1',      1.0, 'all'),
]
SEEDS = [0, 1, 2, 3, 4]
BUDGET = 500

for s in SEEDS:
    for tag, lam, mode in final_models:
        name = f'final_{tag}_s{s}.pt'
        marker = os.path.join(P, name + '.done')
        if os.path.exists(marker):
            print('skip (already done):', name)
            continue
        best = train_model(name, lam=lam, mode=mode, seed=s)
        open(marker, 'w').write(str(best))
        print(f'DONE {name} | best validation accuracy {best:.3f}')

## 9. FINAL RUNS, part 2: the test set and the attacks
The test set is loaded here for the first time. For each seed, the attack runs on the test "no" clips that all five models of that seed classify correctly. Strategies: random, random-yes, near-threshold. Pulses: 1.0 and 0.5 x threshold. Budget: up to 500 spikes.

In [ ]:
T, H = 81, 256

# the test set is loaded, after all models are trained
Xte, yte = torch.load('yesno_testing.pt')
Xte = norm(Xte)

def run_cur(m, x, inj, pulse, record=False):
    mem1 = m.lif1.init_leaky(); mem2 = m.lif2.init_leaky()
    counts = torch.zeros(x.shape[0], 2, device=x.device)
    mems, spks = [], []
    for t in range(x.shape[2]):
        cur1 = m.fc1(x[:, :, t]) + inj[:, t, :] * pulse
        spk1, mem1 = m.lif1(cur1, mem1)
        spk2, mem2 = m.lif2(m.fc2(spk1), mem2)
        counts += spk2
        if record:
            mems.append(mem1); spks.append(spk1)
    if record:
        return counts, torch.stack(mems, 1), torch.stack(spks, 1)
    return counts, None, None

def attack_on(m, X, idx, mode, budget, pulse, seed):
    x = X[idx].to(device); B = len(idx)
    d = (m.fc2.weight[1] - m.fc2.weight[0]).detach()
    with torch.no_grad():
        zero = torch.zeros(B, T, H, device=device)
        _, mems, spks = run_cur(m, x, zero, 0.0, record=True)
        if mode == 'random':
            g = torch.Generator(device=device).manual_seed(seed)
            score = torch.rand(B, T, H, generator=g, device=device)
        else:
            if mode == 'random_yes':
                g = torch.Generator(device=device).manual_seed(seed)
                score = torch.rand(B, T, H, generator=g, device=device)
            else:
                score = mems.clone()
            score[spks > 0] = -1e9
            score[:, :, d <= 0] = -1e9
        order = score.reshape(B, -1).topk(budget, dim=1).indices
        inj = torch.zeros(B, T * H, device=device)
        crit = torch.full((B,), float('inf'))
        for k in range(budget + 1):
            if k > 0:
                inj.scatter_(1, order[:, k-1:k], 1.0)
            counts, _, _ = run_cur(m, x, inj.view(B, T, H), pulse)
            flipped = (counts[:, 1] > counts[:, 0]).cpu()
            crit[flipped & torch.isinf(crit)] = k
            if not torch.isinf(crit).any():
                break
    return crit

def predict_on(m, X):
    m.eval()
    with torch.no_grad():
        return torch.cat([m(X[i:i+256].to(device)).sum(0).argmax(1).cpu() for i in range(0, len(X), 256)])

def activity_on(m, X):
    hs = os_ = 0.0
    with torch.no_grad():
        for i in range(0, len(X), 128):
            xb = X[i:i+128].to(device)
            counts, _, spks = run_cur(m, xb, torch.zeros(len(xb), T, H, device=device), 0.0, record=True)
            hs += spks.sum().item(); os_ += counts.sum().item()
    return hs / len(X), os_ / len(X)

for s in SEEDS:
    out_fn = os.path.join(P, f'final_results_seed{s}.pt')
    if os.path.exists(out_fn):
        print('skip (already done): seed', s)
        continue
    nets_s = {}
    for tag, lam, mode_ in final_models:
        name = f'final_{tag}_s{s}.pt'
        if not os.path.exists(name):
            shutil.copy(os.path.join(P, name), name)
        mm = SNN().to(device); mm.load_state_dict(torch.load(name)); mm.eval(); nets_s[tag] = mm
    ok = None
    res = {'clean_acc': {}, 'activity': {}, 'crit': {}}
    for tag, mm in nets_s.items():
        p_ = predict_on(mm, Xte)
        res['clean_acc'][tag] = (p_ == yte).float().mean().item()
        res['activity'][tag] = activity_on(mm, Xte)
        c = (yte == 0) & (p_ == 0)
        ok = c if ok is None else (ok & c)
    idx_s = ok.nonzero().squeeze(1)
    res['n_clips'] = len(idx_s)
    for pulse in (1.0, 0.5):
        for tag, mm in nets_s.items():
            for mode in ('random', 'random_yes', 'near'):
                res['crit'][(pulse, tag, mode)] = attack_on(mm, Xte, idx_s, mode, BUDGET, pulse, s)
    torch.save(res, out_fn)
    print(f'seed {s} done | shared correctly rejected test "no" clips: {len(idx_s)} | clean test accuracy:',
          {t: round(v, 3) for t, v in res['clean_acc'].items()})

## 10. FINAL RUNS, part 3: statistics and CSV files

In [ ]:
R = {}
for s in SEEDS:
    r = torch.load(os.path.join(P, f'final_results_seed{s}.pt'), weights_only=False)
    r['crit'] = {k: v.numpy() for k, v in r['crit'].items()}
    R[s] = r
print('loaded seeds:', list(R.keys()))

import numpy as np, csv
from scipy import stats

TAGS = [t for t, _, _ in final_models]
GRID = np.arange(0, BUDGET + 1, 5)

def asr(c, b): return float((c <= b).mean())

def ci95(vals):
    v = np.array(vals, float); n = len(v); m = v.mean()
    if n < 2: return m, m, m
    h = stats.t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)
    return m, m - h, m + h

def crit25(c):
    k = int(np.ceil(0.25 * len(c)))
    srt = np.sort(c)
    return float(srt[k - 1]) if np.isfinite(srt[k - 1]) else float('inf')

def mcnemar(a, b):
    n01 = int((a & ~b).sum()); n10 = int((~a & b).sum()); n = n01 + n10
    p = 1.0 if n == 0 else float(stats.binomtest(min(n01, n10), n, 0.5).pvalue)
    return p, n01, n10

# clean test accuracy and activity
print('=== clean test accuracy (mean, 95% CI over seeds) and activity per clip (mean over seeds) ===')
summary_rows = []
for tag in TAGS:
    acc = ci95([R[s]['clean_acc'][tag] for s in SEEDS])
    hid = np.mean([R[s]['activity'][tag][0] for s in SEEDS])
    out = np.mean([R[s]['activity'][tag][1] for s in SEEDS])
    print(f'{tag:14s} accuracy {acc[0]:.3f} [{acc[1]:.3f}, {acc[2]:.3f}] | hidden spikes {hid:.0f} | output spikes {out:.1f} | hidden->output synaptic ops {2*hid:.0f}')
    summary_rows.append([tag, acc[0], acc[1], acc[2], hid, out, 2 * hid, 40 * 256 * 81])
print('dense first-layer multiply-accumulates per clip (all models):', 40 * 256 * 81)
print('shared test "no" clips per seed:', [R[s]['n_clips'] for s in SEEDS])

# attack success rate and critical spike budget
print('\n=== attack success rate, mean [95% CI] over seeds ===')
curve_rows = []
for pulse in (1.0, 0.5):
    print(f'--- pulse = {pulse} x threshold')
    for tag in TAGS:
        for mode in ('random', 'random_yes', 'near'):
            cs = [R[s]['crit'][(pulse, tag, mode)] for s in SEEDS]
            at = {b: ci95([asr(c, b) for c in cs]) for b in (50, 200, 500)}
            cb = [crit25(c) for c in cs]
            cb_txt = ['>500' if not np.isfinite(v) else str(int(v)) for v in cb]
            print(f'{tag:14s} {mode:10s} ' + ' | '.join(f'@{b}: {at[b][0]:.3f} [{at[b][1]:.3f}, {at[b][2]:.3f}]' for b in at) + f' | 25% budget per seed: {cb_txt}')
            for b in GRID:
                m_, lo, hi = ci95([asr(c, int(b)) for c in cs])
                curve_rows.append([pulse, tag, mode, int(b), m_, lo, hi])

# paired tests per seed (McNemar, exact), pulse = 1.0
print('\n=== H1: near-threshold vs random-yes (pulse = theta). Seeds with p < 0.05, and seeds where near flips more ===')
for tag in TAGS:
    for b in (50, 200, 500):
        sig = more = 0
        for s in SEEDS:
            a = R[s]['crit'][(1.0, tag, 'near')] <= b
            c = R[s]['crit'][(1.0, tag, 'random_yes')] <= b
            p, n01, n10 = mcnemar(a, c)
            sig += p < 0.05; more += n01 > n10
        print(f'{tag:14s} budget {b:3d}: significant in {sig}/5 seeds | near flips more in {more}/5 seeds')

print('\n=== H2: guarded vs baseline under the near-threshold attack (pulse = theta) ===')
for tag in TAGS[1:]:
    for b in (50, 200, 500):
        sig = more = 0
        for s in SEEDS:
            g_ = R[s]['crit'][(1.0, tag, 'near')] <= b
            b_ = R[s]['crit'][(1.0, 'baseline', 'near')] <= b
            p, n01, n10 = mcnemar(g_, b_)
            sig += p < 0.05; more += n01 > n10
        print(f'{tag:14s} budget {b:3d}: significant in {sig}/5 seeds | guarded flips more in {more}/5 seeds')

# CSV files for the graphs and tables
for fn in ('final_asr_curves.csv', 'final_summary.csv'):
    pass
with open(os.path.join(P, 'final_asr_curves.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['pulse', 'model', 'mode', 'budget', 'mean', 'ci_low', 'ci_high']); w.writerows(curve_rows)
with open(os.path.join(P, 'final_summary.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['model', 'acc_mean', 'acc_ci_low', 'acc_ci_high', 'hidden_spikes', 'output_spikes', 'hidden_to_output_synaptic_ops', 'dense_first_layer_macs']); w.writerows(summary_rows)
print('\nsaved final_asr_curves.csv and final_summary.csv to Drive')

## 11. FINAL RUNS, part 4: figures


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'], 'font.size': 11})

style = {'baseline': ('-', 'o', 'black'), 'hidden_lam0.1': ('--', 's', 'black'), 'hidden_lam1': (':', '^', 'black'),
         'hidden_lam10': ('-.', 'D', 'dimgray'), 'all_lam1': ((0, (5, 1)), 'v', 'gray')}
label = {'baseline': 'Baseline', 'hidden_lam0.1': 'Hidden-only, $\\lambda$=0.1', 'hidden_lam1': 'Hidden-only, $\\lambda$=1',
         'hidden_lam10': 'Hidden-only, $\\lambda$=10', 'all_lam1': 'All-neuron, $\\lambda$=1'}

# Figure 1: attack success rate versus injected spikes (near-threshold, pulse = theta), mean and 95% CI over seeds
fig, ax = plt.subplots(figsize=(6.4, 4.2))
for tag in TAGS:
    ls, mk, col = style[tag]
    mean = np.array([ci95([asr(R[s]['crit'][(1.0, tag, 'near')], int(b)) for s in SEEDS])[0] for b in GRID])
    lo = np.array([ci95([asr(R[s]['crit'][(1.0, tag, 'near')], int(b)) for s in SEEDS])[1] for b in GRID])
    hi = np.array([ci95([asr(R[s]['crit'][(1.0, tag, 'near')], int(b)) for s in SEEDS])[2] for b in GRID])
    ax.plot(GRID, mean, linestyle=ls, marker=mk, markevery=10, color=col, markersize=5, label=label[tag])
    ax.fill_between(GRID, lo, hi, color='gray', alpha=0.15, linewidth=0)
ax.set_xlabel('Number of injected spikes')
ax.set_ylabel('Attack success rate')
ax.set_ylim(0, 1.02)
ax.grid(True, color='lightgray', linewidth=0.5)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(P, 'fig_asr_vs_spikes.pdf')); fig.savefig('fig_asr_vs_spikes.pdf')
plt.show()

# Figure 2: near-threshold versus random-yes at 200 spikes (pulse = theta)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
x = np.arange(len(TAGS)); wd = 0.38
for j, (mode, hatch, fc) in enumerate([('near', '//', '0.35'), ('random_yes', '', '0.8')]):
    vals = [ci95([asr(R[s]['crit'][(1.0, tag, mode)], 200) for s in SEEDS]) for tag in TAGS]
    m_ = np.array([v[0] for v in vals]); err = np.array([[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]])
    ax.bar(x + (j - 0.5) * wd, m_, wd, yerr=err, capsize=3, color=fc, edgecolor='black', hatch=hatch,
           label='Near-threshold' if mode == 'near' else 'Random-yes')
ax.set_xticks(x); ax.set_xticklabels([label[t] for t in TAGS], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Attack success rate at 200 spikes')
ax.set_ylim(0, 1.02)
ax.grid(True, axis='y', color='lightgray', linewidth=0.5)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(P, 'fig_near_vs_randomyes.pdf')); fig.savefig('fig_near_vs_randomyes.pdf')
plt.show()

try:
    from google.colab import files
    files.download('fig_asr_vs_spikes.pdf'); files.download('fig_near_vs_randomyes.pdf')
except ImportError:
    pass
print('figures saved to Drive and downloaded')